# Tome C2, Chapter 11: String Algorithms (Python companion)

Exercises first, solutions at the end. We implement the chapter: the failure function and the Knuth–Morris–Pratt search with exact comparison counts, the $Z$-algorithm, Karp–Rabin fingerprints and their false positives, the Aho–Corasick automaton, suffix arrays by prefix doubling with `numpy` and the LCP array of Kasai et al., the Burrows–Wheeler transform with its inversion, backward search and a look at why it compresses, and finally HyperLogLog.

*Before each solution, predict what the output will look like.*

In [ ]:
import random, math, time, bz2, zlib, hashlib, collections
import numpy as np
import matplotlib.pyplot as plt
random.seed(11); rng = np.random.default_rng(11)

## Exercises

**P11.1 (★) Borders.** Implement the failure function (Proposition 11.3) and the KMP search (Theorem 11.4), counting letter comparisons; compare with the naive search on random texts and on the worst case $T=\texttt a^n$, $P=\texttt a^{m-1}\texttt b$. Check Exercise 11.1, and compute the correlation $O(X,X)$ of Penney's game (Tome C1, Chapter 4) from the borders.

**P11.2 (★) $Z$.** Implement the $Z$-algorithm (Theorem 11.5), check it against the definition, use it for matching and for the smallest period (Exercise 11.4).

**P11.3 (★) Fingerprints.** Implement Karp–Rabin modulo the prime $2^{61}-1$, and then with a deliberately small prime, to watch false positives appear at the rate of Theorem 11.6.

**P11.4 (★★) A dictionary.** Implement Aho–Corasick (Theorem 11.7), check it on the example $\{\texttt{he},\texttt{she},\texttt{his},\texttt{hers}\}$ and against naive search for a dictionary of a thousand random words, and compare running times.

**P11.5 (★★) Suffix arrays.** Implement prefix doubling (Theorem 11.9) with `numpy`, and Kasai's LCP algorithm (Theorem 11.10). Check them against sorting, time them on a random DNA string of a million letters, and use them to find the longest repeated factor and the number of distinct factors (Exercise 11.7).

**P11.6 (★★) Burrows–Wheeler.** Compute the transform from the suffix array, invert it with the last-to-first map (Theorem 11.12), count occurrences by backward search, and measure how the transform groups letters on a real text: runs, move-to-front, entropy, and a comparison with `bz2` and `zlib`.

**P11.7 (★★) HyperLogLog.** Implement HyperLogLog with $\mu$ registers, measure its relative error against $1.04/\sqrt\mu$, and check that sketches merge (Exercise 11.10).

## Solutions

### P11.1 Borders

In [ ]:
def failure(P):
    m = len(P); pi = [0] * (m + 1); k = 0; comps = 0
    for i in range(1, m):
        while k > 0:
            comps += 1
            if P[k] == P[i]: break
            k = pi[k]
        else:
            comps += 1
        if P[k] == P[i]: k += 1
        pi[i + 1] = k
    return pi, comps

def kmp(T, P, pi=None):
    m = len(P)
    if pi is None: pi = failure(P)[0]
    q = 0; occ = []; comps = 0
    for j, a in enumerate(T):
        while q == m or (q > 0 and (comps := comps + 1) and P[q] != a):
            q = pi[q]
        comps += 1
        if P[q] == a: q += 1
        if q == m: occ.append(j - m + 1)
    return occ, comps

def naive(T, P):
    occ, comps = [], 0
    for i in range(len(T) - len(P) + 1):
        for k in range(len(P)):
            comps += 1
            if T[i + k] != P[k]: break
        else:
            occ.append(i)
    return occ, comps

print("pi(abacabab) =", failure("abacabab")[0][1:])
print("occurrences in abacabacabab:", kmp("abacabacabab", "abacabab")[0])
for _ in range(3):
    T = "".join(random.choice("ab") for _ in range(20000)); P = "".join(random.choice("ab") for _ in range(8))
    (o1, c1), (o2, c2) = kmp(T, P), naive(T, P)
    print(f"random binary text: {len(o1)} occurrences, agree {o1 == o2}; comparisons KMP {c1} (< 2n = {2*len(T)}), naive {c2}")
n, m = 20000, 50
T, P = "a" * n, "a" * (m - 1) + "b"
print(f"worst case a^n, a^(m-1)b: KMP {kmp(T, P)[1]} comparisons, naive {naive(T, P)[1]}")

The comparison count of KMP stays below $2n$ in every case, while the naive search pays about $nm$ on the worst case. On random binary text the naive search is not bad either: a mismatch comes after two comparisons on average. Now Penney's correlation: $O(X,X)$ is the set of border lengths together with $|X|$, read off the chain $\pi(m),\pi(\pi(m)),\dots$

In [ ]:
def correlation(X):
    pi = failure(X)[0]; b = pi[len(X)]; O = {len(X)}
    while b > 0: O.add(b); b = pi[b]
    return sorted(O)

for X in ("HH", "HT", "HHH", "HTH", "THH", "HTHT", "abacabab"):
    O = correlation(X)
    print(f"O({X},{X}) = {O},  XX = sum of 2^j = {sum(2**j for j in O)},  periods = {[len(X) - j for j in O if j < len(X)] + [len(X)]}")

### P11.2 $Z$

In [ ]:
def zarray(S):
    N = len(S); Z = [0] * N; l = r = 0
    for i in range(1, N):
        if i < r and Z[i - l] < r - i:
            Z[i] = Z[i - l]
        else:
            z = max(0, r - i)
            while i + z < N and S[z] == S[i + z]: z += 1
            Z[i] = z
            if i + z > r: l, r = i, i + z
    return Z

def z_naive(S):
    Z = [0] * len(S)
    for i in range(1, len(S)):
        while i + Z[i] < len(S) and S[Z[i]] == S[i + Z[i]]: Z[i] += 1
    return Z

ok = all(zarray(s) == z_naive(s) for s in ("".join(random.choice("ab") for _ in range(random.randint(1, 60))) for _ in range(3000)))
print("Z agrees with the definition on 3000 random strings:", ok)
T = "".join(random.choice("acgt") for _ in range(10000)); P = T[5000:5012]
Z = zarray(P + "#" + T)
print("Z-matching:", [i for i in range(len(T)) if Z[len(P) + 1 + i] == len(P)], " KMP:", kmp(T, P)[0])
def smallest_period(S):
    Z = zarray(S); N = len(S)
    return next((p for p in range(1, N) if p + Z[p] == N), N)
for S in ("abcabcabc", "abcabcab", "aaaa", "abaab", "abaababaab"):
    p = smallest_period(S)
    print(f"{S}: smallest period {p}, a proper power: {p < len(S) and len(S) % p == 0}")

### P11.3 Fingerprints

In [ ]:
def karp_rabin(T, P, p, x):
    m, n = len(P), len(T)
    if m > n: return [], 0
    enc = lambda c: ord(c) % p
    hp = 0
    for c in P: hp = (hp * x + enc(c)) % p
    h = 0
    for c in T[:m]: h = (h * x + enc(c)) % p
    xm = pow(x, m - 1, p)
    reported = []
    for i in range(n - m + 1):
        if h == hp: reported.append(i)
        if i + m < n: h = ((h - enc(T[i]) * xm) * x + enc(T[i + m])) % p
    true = [i for i in reported if T[i:i + m] == P]
    return true, len(reported) - len(true)

T = "".join(random.choice("acgt") for _ in range(200000)); P = T[123456:123456 + 20]
p = 2**61 - 1
occ, false = karp_rabin(T, P, p, random.randrange(1, p))
print(f"p = 2^61-1: occurrences {occ}, false reports {false}")
for p in (101, 1009, 10007):
    falses = [karp_rabin(T, P, p, random.randrange(1, p))[1] for _ in range(5)]
    print(f"p = {p}: false reports in 5 runs {falses}; bound (n-m+1)(m-1)/p = {(len(T) - 19) * 19 / p:.0f}")

With $p=2^{61}-1$ there is no false report. With small primes they appear, and their number is of the order of the bound $(n-m+1)(m-1)/p$, usually well below it: the bound counts the worst case of a polynomial with $m-1$ roots, while a random difference polynomial has on average about one root.

### P11.4 A dictionary

In [ ]:
class AhoCorasick:
    def __init__(self, words):
        self.goto = [{}]; self.out = [None]; self.fail = [0]; self.olink = [None]
        for w in words:
            v = 0
            for c in w:
                if c not in self.goto[v]:
                    self.goto.append({}); self.out.append(None); self.fail.append(0); self.olink.append(None)
                    self.goto[v][c] = len(self.goto) - 1
                v = self.goto[v][c]
            self.out[v] = w
        queue = collections.deque(self.goto[0].values())
        while queue:
            v = queue.popleft()
            for c, u in self.goto[v].items():
                queue.append(u)
                f = self.fail[v]
                while f and c not in self.goto[f]: f = self.fail[f]
                self.fail[u] = self.goto[f][c] if c in self.goto[f] and self.goto[f][c] != u else 0
                fu = self.fail[u]
                self.olink[u] = fu if self.out[fu] is not None else self.olink[fu]
    def search(self, T):
        v = 0; res = []
        for j, c in enumerate(T):
            while v and c not in self.goto[v]: v = self.fail[v]
            v = self.goto[v].get(c, 0)
            u = v if self.out[v] is not None else self.olink[v]
            while u is not None and u != 0:
                res.append((j - len(self.out[u]) + 1, self.out[u])); u = self.olink[u]
        return res

ac = AhoCorasick(["he", "she", "his", "hers"])
print("failure links:", {i: ac.fail[i] for i in range(1, len(ac.fail))})
print("ushers:", ac.search("ushers"))
words = list({"".join(random.choice("abcdefgh") for _ in range(random.randint(3, 7))) for _ in range(1000)})
T = "".join(random.choice("abcdefgh") for _ in range(100000))
t0 = time.perf_counter(); A = AhoCorasick(words); res = A.search(T); t1 = time.perf_counter() - t0
t0 = time.perf_counter(); res2 = [(i, w) for w in words for i in kmp(T, w)[0]]; t2 = time.perf_counter() - t0
print(f"{len(words)} words, text of {len(T)} letters: {len(res)} occurrences; agree with one KMP per word: {sorted(res) == sorted(res2)}")
print(f"Aho-Corasick {t1:.2f} s, KMP word by word {t2:.1f} s")

The automaton reads the text once for all the words, while separate searches read it once per word: it is hundreds of times faster here. The failure links agree with Figure 11.1 ($4\to1$, $5\to2$, $7\to3$, $9\to3$, the others to the root), and `ushers` contains `she`, `he` and `hers`.

### P11.5 Suffix arrays

In [ ]:
def suffix_array(s):
    # prefix doubling with numpy: ranks of factors of length 2^k, sorted as pairs (Theorem 11.9).
    # The last character is the sentinel and gets code 0, smaller than every other letter.
    a = np.array([ord(c) + 1 for c in s], dtype=np.int64); a[-1] = 0
    N = len(a); rank = a.copy(); k = 1
    while True:
        second = np.full(N, -1, dtype=np.int64); second[:N - k] = rank[k:]
        order = np.lexsort((second, rank))
        r1, r2 = rank[order], second[order]
        new = np.empty(N, dtype=np.int64)
        new[order] = np.concatenate(([0], np.cumsum((r1[1:] != r1[:-1]) | (r2[1:] != r2[:-1]))))
        rank = new
        if rank.max() == N - 1: return order
        k *= 2

def kasai(s, sa):
    N = len(s); rk = [0] * N
    for k, i in enumerate(sa): rk[i] = k
    lcp = [0] * N; h = 0
    for i in range(N):
        if rk[i] > 0:
            j = sa[rk[i] - 1]
            while i + h < N and j + h < N and s[i + h] == s[j + h]: h += 1
            lcp[rk[i]] = h
            if h: h -= 1
        else:
            h = 0
    return lcp

s = "banana$"
print("SA(banana$) =", suffix_array(s).tolist(), " LCP =", kasai(s, suffix_array(s).tolist()))
for _ in range(300):
    t = "".join(random.choice("ab") for _ in range(random.randint(1, 80))) + "$"
    assert list(suffix_array(t)) == sorted(range(len(t)), key=lambda i: t[i:])
print("prefix doubling agrees with sorting on 300 random strings")
dna = "".join(rng.choice(list("acgt"), size=10**6)) + "$"
t0 = time.perf_counter(); sa = suffix_array(dna); t1 = time.perf_counter() - t0
t0 = time.perf_counter(); lcp = kasai(dna, sa.tolist()); t2 = time.perf_counter() - t0
best = int(np.argmax(lcp))
print(f"10^6 random letters: suffix array {t1:.1f} s, LCP {t2:.1f} s; longest repeat has length {lcp[best]} (log_4 of n^2/2 is {math.log(1e12/2, 4):.1f})")
w = "banana"; sw = w + "$"; L = kasai(sw, list(suffix_array(sw)))
print("distinct factors of banana:", len(w) * (len(w) + 1) // 2 - sum(L), "by brute force:", len({w[i:j] for i in range(len(w)) for j in range(i + 1, len(w) + 1)}))

In a random string over four letters the longest repeat has length close to $\log_4(n^2/2)$, about $19$ for $n=10^6$: a repeated factor of length $\ell$ is expected about $\binom n2 4^{-\ell}$ times, and this crosses $1$ there. A real genome has repeats of thousands of letters, which is exactly what makes assembling it hard.

### P11.6 Burrows–Wheeler

In [ ]:
def bwt(s):
    sa = suffix_array(s)
    return "".join(s[i - 1] for i in sa), sa

def inverse_bwt(L):
    C, counts = {}, collections.Counter(L); tot = 0
    for c in sorted(counts, key=lambda c: (c != "$", c)): C[c] = tot; tot += counts[c]
    seen = collections.Counter(); LF = []
    for c in L: LF.append(C[c] + seen[c]); seen[c] += 1
    out = []; k = 0
    for _ in range(len(L) - 1):
        out.append(L[k]); k = LF[k]
    return "".join(reversed(out)) + "$"

def backward_count(L, C, occ, P):
    s, e = 0, len(L)
    for c in reversed(P):
        if c not in C: return 0
        s, e = C[c] + occ[c][s], C[c] + occ[c][e]
        if s >= e: return 0
    return e - s

L, sa = bwt("banana$"); print("BWT(banana$) =", L, " inverse:", inverse_bwt(L))
text = open(random.__file__).read().replace("$", "") + "$"     # Python's own random.py as a real text
L, sa = bwt(text)
print(f"text of {len(text)} letters; inversion correct: {inverse_bwt(L) == text}")
counts = collections.Counter(L); C = {}; tot = 0
for c in sorted(counts, key=lambda c: (c != "$", c)): C[c] = tot; tot += counts[c]
codes = np.array([ord(c) for c in L])
occ = {c: np.concatenate(([0], np.cumsum(codes == ord(c)))) for c in counts}    # rank tables
for P in ("random", "def ", "return", "self.", "Mersenne"):
    print(f"{P!r}: backward search {backward_count(L, C, occ, P)}, str.count-like {sum(1 for i in range(len(text)) if text.startswith(P, i))}")

In [ ]:
def runs(s): return 1 + sum(s[i] != s[i - 1] for i in range(1, len(s)))
def mtf(s):
    alphabet = sorted(set(s)); out = []
    for c in s:
        i = alphabet.index(c); out.append(i); alphabet.insert(0, alphabet.pop(i))
    return out
def entropy(seq):
    cnt = collections.Counter(seq); n = len(seq)
    return -sum(v / n * math.log2(v / n) for v in cnt.values())
mt_text, mt_L = mtf(text), mtf(L)
print(f"runs of equal letters: text {runs(text)}, BWT {runs(L)}")
print(f"zeros after move-to-front: text {mt_text.count(0)/len(text):.1%}, BWT {mt_L.count(0)/len(L):.1%}")
print(f"order-0 entropy (bits per letter): text {entropy(text):.2f}, MTF of text {entropy(mt_text):.2f}, MTF of BWT {entropy(mt_L):.2f}")
raw = text.encode()
print(f"bz2: {8*len(bz2.compress(raw))/len(raw):.2f} bits per letter, zlib: {8*len(zlib.compress(raw, 9))/len(raw):.2f}")
plt.figure(figsize=(8, 3))
plt.hist([mt_text, mt_L], bins=range(0, 40), label=["MTF of text", "MTF of BWT"], log=True)
plt.xlabel("move-to-front index"); plt.legend(); plt.tight_layout(); plt.show()

The transform halves the number of runs of equal letters, and after move-to-front most of its symbols are $0$ (the letter just seen), and the order-$0$ entropy of the result is far below that of the text. A simple entropy coder on this sequence already beats letter-by-letter coding, and `bz2`, which is exactly block sorting followed by move-to-front and Huffman coding, compresses this file better than `zlib` (Lempel–Ziv with Huffman). The backward search counts agree with direct counting, and each count used two rank lookups per letter of the pattern, whatever the length of the text.

### P11.7 HyperLogLog

In [ ]:
class HLL:
    def __init__(self, b):
        self.b = b; self.mu = 1 << b; self.M = np.zeros(self.mu, dtype=np.int64)
        self.alpha = 0.7213 / (1 + 1.079 / self.mu)
    def add(self, item):
        h = int.from_bytes(hashlib.blake2b(item.encode(), digest_size=8).digest(), "big")
        j = h >> (64 - self.b); w = h & ((1 << (64 - self.b)) - 1)
        rho = (64 - self.b) - w.bit_length() + 1
        if rho > self.M[j]: self.M[j] = rho
    def estimate(self):
        E = self.alpha * self.mu**2 / np.sum(2.0 ** (-self.M))
        V = int(np.sum(self.M == 0))
        if E <= 2.5 * self.mu and V > 0: E = self.mu * math.log(self.mu / V)   # small-range correction of the paper
        return E
    def merge(self, other):
        out = HLL(self.b); out.M = np.maximum(self.M, other.M); return out

F = 100000
for b in (6, 8, 10):
    errs = []
    for trial in range(20):
        h = HLL(b)
        for i in range(F): h.add(f"{trial}-{i}")
        errs.append(h.estimate() / F - 1)
    print(f"mu = {1<<b:5d} registers ({(1<<b)*6/8:.0f} bytes): relative error std {np.std(errs):.3f}, 1.04/sqrt(mu) = {1.04/math.sqrt(1<<b):.3f}")
A, B, U = HLL(10), HLL(10), HLL(10)
for i in range(60000): A.add(f"x{i}"); U.add(f"x{i}")
for i in range(30000, 100000): B.add(f"x{i}"); U.add(f"x{i}")
print(f"merge: max of registers equals the sketch of the union: {np.array_equal(A.merge(B).M, U.M)}; estimate {A.merge(B).estimate():.0f} for 100000")

The measured relative errors are of the size $1.04/\sqrt\mu$ predicted by Flajolet and his coauthors (over twenty trials, the estimate of a standard deviation is itself only accurate to about $15\%$). A kilobyte of registers counts a hundred thousand distinct strings to within a few per cent, and the merged sketch is identical to the sketch of the union, bit for bit.